In [0]:
from pyspark.sql.functions import (
    col, to_date, weekofyear, year, hour, when,
    sum as _sum, avg, round, desc
)

# 1. Read CSV from your training_files Volume
volume_path = "/Volumes/workspace/default/training_files/large_sensor_logs.csv"

raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(volume_path)
)

print(f"Total sensor records loaded: {raw_df.count()}")
display(raw_df.limit(5))

Total sensor records loaded: 21


device_id,device_type,room_id,timestamp,energy_kwh,voltage
DEV_HVAC_01,HVAC,1,2026-10-08T02:00:00.000Z,0.45,230.1
DEV_HVAC_01,HVAC,1,2026-10-08T07:30:00.000Z,1.85,229.4
DEV_HVAC_01,HVAC,1,2026-10-08T14:15:00.000Z,2.4,228.9
DEV_HVAC_01,HVAC,1,2026-10-08T20:00:00.000Z,2.15,231.0
DEV_HVAC_01,HVAC,1,2026-10-08T23:30:00.000Z,0.52,230.5


In [0]:
# 1. Enrich with Date, Week, Year, and Peak/Off-Peak Classification
enriched_df = (
    raw_df
    .withColumn("usage_date", to_date(col("timestamp")))
    .withColumn("usage_week", weekofyear(col("timestamp")))
    .withColumn("usage_year", year(col("timestamp")))
    .withColumn("reading_hour", hour(col("timestamp")))
    .withColumn(
        "tariff_zone",
        when((col("reading_hour") >= 6) & (col("reading_hour") < 22), "PEAK")
        .otherwise("OFF_PEAK")
    )
)

# 2. Daily Summary per Device
daily_summary_df = (
    enriched_df
    .groupBy("usage_date", "device_id", "device_type", "room_id")
    .agg(
        round(_sum("energy_kwh"), 3).alias("daily_total_kwh"),
        round(avg("voltage"), 1).alias("avg_voltage"),
        round(_sum(when(col("tariff_zone") == "PEAK", col("energy_kwh")).otherwise(0)), 3).alias("peak_kwh"),
        round(_sum(when(col("tariff_zone") == "OFF_PEAK", col("energy_kwh")).otherwise(0)), 3).alias("off_peak_kwh")
    )
    .orderBy("usage_date", desc("daily_total_kwh"))
)

print("--- Daily Energy Summary ---")
display(daily_summary_df)

# 3. Weekly Summary per Device
weekly_summary_df = (
    enriched_df
    .groupBy("usage_year", "usage_week", "device_id", "device_type")
    .agg(
        round(_sum("energy_kwh"), 3).alias("weekly_total_kwh"),
        round(avg("energy_kwh"), 3).alias("avg_reading_kwh")
    )
    .orderBy("usage_year", "usage_week", desc("weekly_total_kwh"))
)

print("--- Weekly Energy Summary ---")
display(weekly_summary_df)

--- Daily Energy Summary ---


usage_date,device_id,device_type,room_id,daily_total_kwh,avg_voltage,peak_kwh,off_peak_kwh
2026-10-08,DEV_EV_01,EV_Charger,4,15.25,231.3,1.2,14.05
2026-10-08,DEV_HVAC_01,HVAC,1,7.37,230.0,6.4,0.97
2026-10-08,DEV_HEATER_01,Heating,3,5.1,230.5,2.45,2.65
2026-10-08,DEV_WASH_01,Laundry,2,1.82,229.5,1.8,0.02
2026-10-08,DEV_FRIDGE_01,Refrigeration,2,0.668,230.1,0.433,0.235


--- Weekly Energy Summary ---


usage_year,usage_week,device_id,device_type,weekly_total_kwh,avg_reading_kwh
2026,41,DEV_EV_01,EV_Charger,15.25,5.083
2026,41,DEV_HVAC_01,HVAC,7.37,1.474
2026,41,DEV_HEATER_01,Heating,5.1,1.275
2026,41,DEV_WASH_01,Laundry,1.82,0.455
2026,41,DEV_FRIDGE_01,Refrigeration,0.668,0.134


In [0]:
# 1. Save as Managed Delta Tables in workspace.default
daily_summary_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.smart_home_daily_summary")
weekly_summary_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.smart_home_weekly_summary")

print("[SUCCESS] Saved Delta tables: smart_home_daily_summary & smart_home_weekly_summary")

# 2. Save clean CSV files into training_files volume for dashboards/alerts
daily_summary_df.toPandas().to_csv("/Volumes/workspace/default/training_files/output_daily_summary.csv", index=False)
weekly_summary_df.toPandas().to_csv("/Volumes/workspace/default/training_files/output_weekly_summary.csv", index=False)

print("[SUCCESS] Exported CSV output files to /Volumes/workspace/default/training_files/")

[SUCCESS] Saved Delta tables: smart_home_daily_summary & smart_home_weekly_summary
[SUCCESS] Exported CSV output files to /Volumes/workspace/default/training_files/


In [0]:
%sql
SELECT 
    device_id,
    device_type,
    ROUND(SUM(daily_total_kwh), 2) AS total_kwh,
    ROUND(SUM(peak_kwh), 2) AS total_peak_kwh,
    ROUND((SUM(peak_kwh) / SUM(daily_total_kwh)) * 100, 1) AS peak_usage_pct,
    CASE 
        WHEN (SUM(peak_kwh) / SUM(daily_total_kwh)) > 0.60 THEN 'HIGH PEAK ALERT: Shift schedule to Off-Peak'
        ELSE 'OPTIMAL: Balanced Tariff Usage'
    END AS savings_recommendation
FROM workspace.default.smart_home_daily_summary
GROUP BY device_id, device_type
ORDER BY total_peak_kwh DESC;

device_id,device_type,total_kwh,total_peak_kwh,peak_usage_pct,savings_recommendation
DEV_HVAC_01,HVAC,7.37,6.4,86.8,HIGH PEAK ALERT: Shift schedule to Off-Peak
DEV_HEATER_01,Heating,5.1,2.45,48.0,OPTIMAL: Balanced Tariff Usage
DEV_WASH_01,Laundry,1.82,1.8,98.9,HIGH PEAK ALERT: Shift schedule to Off-Peak
DEV_EV_01,EV_Charger,15.25,1.2,7.9,OPTIMAL: Balanced Tariff Usage
DEV_FRIDGE_01,Refrigeration,0.67,0.43,64.8,HIGH PEAK ALERT: Shift schedule to Off-Peak
